# 第 3 章 線形回帰

部屋数から住宅価格を予測する線形回帰を、simple / absolute / square の 3 つのトリックで学習します。

対応する記事: [第 3 章 線形回帰（Kotlin 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/kotlin/ch03.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch03.*

## データセット

原著と同じ 6 件の住宅データです。特徴量は部屋数、ラベルは価格です。

In [2]:
val features = listOf(1.0, 2.0, 3.0, 5.0, 6.0, 7.0)
val labels = listOf(155.0, 197.0, 244.0, 356.0, 407.0, 448.0)

features.zip(labels).forEach { (x, y) -> println("部屋数 %.0f → 価格 %.0f".format(x, y)) }

部屋数 1 → 価格 155
部屋数 2 → 価格 197
部屋数 3 → 価格 244


部屋数 5 → 価

格 356
部屋数 6 → 価格 407
部屋数 7 → 価格 448


## 3 つのトリックを 1 点分だけ試す

予測が `50 × 3 + 100 = 250`、正解が 300 のとき、それぞれのトリックがどう動くかを見ます。

二乗トリックだけが **誤差の大きさに比例** して動くことに注目してください。

In [3]:
val model = Model(slope = 50.0, intercept = 100.0)
println("予測 ${model.predict(3.0)} 正解 300.0")

println("absolute " + absoluteTrick(model, rooms = 3.0, price = 300.0, learningRate = 0.01))
println("square   " + squareTrick(model, rooms = 3.0, price = 300.0, learningRate = 0.01))

予測 250.0 正解 300.0
absolute Model(slope=50.03, intercept=100.01)


square   Model(slope=51.5, intercept=100.5)


## 学習

学習率 0.01、1000 エポックで学習します。真の関係は「傾き 50・切片 100」です。

In [4]:
val (trained, errors) = linearRegression(features, labels, learningRate = 0.01, epochs = 1000, seed = 0)

println("傾き   %.4f".format(trained.slope))
println("切片   %.4f".format(trained.intercept))
println("RMSE   %.4f".format(modelRmse(trained, features, labels)))
println("初期の RMSE %.2f → 最終 %.4f".format(errors.first(), errors.last()))

傾き   52.4794
切片   88.3304
RMSE   7.2976
初期の RMSE 317.01 → 最終 7.2796


## 誤差の推移

エポックごとの RMSE を 100 エポック刻みで見ます。**最初の数百エポックで大きく下がり、その後は緩やかになります。**

In [5]:
for (epoch in 0 until 1000 step 100) {
    val bar = "#".repeat((errors[epoch] / 8).toInt())
    println("epoch %4d  RMSE %7.3f  %s".format(epoch, errors[epoch], bar))
}

epoch    0  RMSE 317.007  #######################################
epoch  100  RMSE  34.197  ####


epoch  200  RMSE  28.125  ###
epoch  300  RMSE  24.022  ###
epoch  400  RMSE  20.120  ##


epoch  500  RMSE  15.085  #
epoch  600  RMSE  12.562  #
epoch  700  RMSE  11.279  #


epoch  800  RMSE   9.220  #
epoch  900  RMSE   8.811  #


## 予測してみる

学習したモデルで、部屋数 4 の家の価格を予測します。

In [6]:
listOf(1.0, 4.0, 8.0).forEach { rooms ->
    println("部屋数 %.0f → 予測価格 %.2f".format(rooms, trained.predict(rooms)))
}

部屋数 1 → 予測価格 140.81
部屋数 4 → 予測価格 298.25


部屋数 8 → 予測価格 508.17


## 試してみる

学習率を変えると何が起きるでしょうか。**大きすぎると発散し、小さすぎると収束しません。**

In [7]:
listOf(0.001, 0.01, 0.1).forEach { rate ->
    val (m, _) = linearRegression(features, labels, learningRate = rate, epochs = 1000, seed = 0)
    println("学習率 %-6s 傾き %9.4f  RMSE %10.4f".format(rate, m.slope, modelRmse(m, features, labels)))
}

学習率 0.001  傾き   64.9760  RMSE    34.0096


学習率 0.01   傾き   52.4794  RMSE     7.2976


学習率 0.1    傾き 1817408180128362200000000000000000000000000000.0000  RMSE 4326296814611671000000000000000000000000000000.0000
